## Day 1: Agents vs. Chatbots, and the ReAct Loop

A chatbot maps one message to one reply. An **agent** repeats an
observe -> reason -> act cycle until it reaches a goal or hits a stop
condition -- the loop is the entire distinction (see `daily/Day1_...md`
for the full state-machine diagram and the mechanics of the
`stop=["Observation:"]` sequence that keeps observations real instead of
hallucinated).

This section uses a currency-conversion scenario (different from the
daily note's grocery-bill example) to show the same three things end to
end: a plain chatbot guessing wrong, a ReAct loop getting it right via a
real tool call, few-shot priming, and -- critically -- the `max_steps`
guard actually firing when a model never converges to a `Final Answer`.

In [ ]:
FEW_SHOT_PRIMER = '''
Question: How many euros is 100 US dollars?
Thought: I need the current USD->EUR rate before I can answer.
Action: convert_currency(100, "USD", "EUR")
Observation: 92.10
Thought: I now have enough to answer.
Final Answer: About 92.10 EUR.
'''

def convert_currency(amount, src, dst):
    # Stand-in for a real FX-rate lookup tool.
    fake_rates = {("USD", "EUR"): 0.921, ("USD", "JPY"): 149.2}
    rate = fake_rates.get((src, dst), 1.0)
    return round(amount * rate, 2)   # -> float

TOOLS = {"convert_currency": convert_currency}

def parse_action(line):
    # 'Action: convert_currency(100, "USD", "EUR")' -> ("convert_currency", (100.0, "USD", "EUR"))
    name = line.split("Action:")[1].split("(")[0].strip()
    raw_args = line.split("(", 1)[1].rsplit(")", 1)[0]
    parts = [p.strip().strip('"') for p in raw_args.split(",")]
    amount, src, dst = float(parts[0]), parts[1], parts[2]
    return name, (amount, src, dst)

def run_currency_agent(fake_model_generate, question, max_steps=5):
    transcript = FEW_SHOT_PRIMER + f"\nQuestion: {question}\n"
    for step in range(max_steps):  # hard safety cap: never trust the model to stop itself
        chunk = fake_model_generate(transcript)
        transcript += chunk
        if "Final Answer:" in chunk:
            return chunk.split("Final Answer:")[-1].strip()
        action_line = next(l for l in chunk.splitlines() if l.startswith("Action:"))
        name, (amount, src, dst) = parse_action(action_line)
        result = TOOLS[name](amount, src, dst)               # real tool call, real number back
        transcript += f"Observation: {result}\n"             # harness writes this -- never the model
    return "Stopped: exceeded max_steps without a Final Answer."

In [ ]:
# Two mock model behaviors to illustrate the before/after contrast: a
# plain chatbot with no tool access, vs. the same "model" wired into the
# ReAct loop with a real convert_currency tool available.

def plain_chatbot_mock(prompt):
    # A chatbot with no tool access just guesses fluently -- and can be wrong.
    return "Final Answer: Roughly 100 EUR (same as the dollar amount)."

def react_mock_generate(transcript):
    # Deterministic stand-in for a real local model call, e.g.:
    #   inputs = tokenizer(transcript, return_tensors="pt")
    #   out = model.generate(**inputs, max_new_tokens=64, stop_strings=["Observation:"])
    # The stop_strings argument is what prevents the model from writing
    # its own fabricated Observation line -- see the daily note.
    if "Observation:" not in transcript:
        return 'Thought: I need the rate.\nAction: convert_currency(100, "USD", "EUR")\n'
    return "Thought: Done.\nFinal Answer: 92.10 EUR.\n"

print(plain_chatbot_mock("How many euros is 100 US dollars?"))
print(run_currency_agent(react_mock_generate, "How many euros is 100 US dollars?"))

In [ ]:
# The max_steps guard, tested for real (not just described): a mock
# model that never emits Final Answer -- it keeps re-asking the same
# question forever. Confirms the loop actually halts instead of spinning.

def looping_mock_generate(transcript):
    return 'Thought: let me check the rate again.\nAction: convert_currency(100, "USD", "EUR")\n'

print(run_currency_agent(looping_mock_generate, "How many euros is 100 US dollars?", max_steps=3))
# -> "Stopped: exceeded max_steps without a Final Answer." (verified output)

## Day 2: Agent Framework Landscape and a Tiny Framework

The wider agent-tooling ecosystem roughly splits into autonomous
runtimes, sandboxing/security wrappers, and framework-agnostic
orchestration layers on top of several engines (see `daily/Day2_...md`
for the comparison diagram). Rather than reusing the `MiniAgentFramework`
example from the daily note, this section builds a `ToolRegistry` -- tool
bookkeeping, a retry budget for flaky tools, and an audit log -- around a
document-translation scenario, plus a separate LCEL-style pipe chain.

In [ ]:
import time

class ToolRegistry:
    """A minimal stand-in for what larger agent frameworks manage: a table
    of callable tools (with an allow/deny flag and an optional retry
    budget for flaky tools) plus a running audit log."""

    def __init__(self):
        self._tools = {}
        self.audit_log = []

    def register_tool(self, name, fn, allowed=True, max_retries=0):
        self._tools[name] = {"fn": fn, "allowed": allowed, "max_retries": max_retries}

    def run_tool(self, name, **kwargs):
        record = {"tool": name, "kwargs": kwargs, "ts": round(time.time(), 3)}
        entry = self._tools.get(name)
        if entry is None or not entry["allowed"]:
            record["status"] = "denied"
            self.audit_log.append(record)
            raise PermissionError(f"tool '{name}' is not available")

        attempts, last_err = 0, None
        while attempts <= entry["max_retries"]:
            attempts += 1
            try:
                result = entry["fn"](**kwargs)          # -> whatever the tool returns
                record["status"] = "ok"
                record["attempts"] = attempts
                record["result"] = result
                self.audit_log.append(record)
                return result
            except Exception as e:                       # flaky tool call failed; retry if budget remains
                last_err = e
        record["status"] = "failed_after_retries"
        record["attempts"] = attempts
        record["error"] = str(last_err)
        self.audit_log.append(record)
        raise RuntimeError(f"tool '{name}' failed after {attempts} attempts: {last_err}")

def detect_language(text):
    return "fr" if text.lower().startswith("bonjour") else "en"

# A tool that times out on its first call, then succeeds -- exercises the
# retry path for real, not just in description.
_flaky_state = {"n": 0}
def flaky_translate_api(text):
    _flaky_state["n"] += 1
    if _flaky_state["n"] < 2:
        raise ConnectionError(f"simulated timeout on attempt {_flaky_state['n']}")
    return f"[ES] {text}"

registry = ToolRegistry()
registry.register_tool("detect_language", detect_language)
registry.register_tool("translate", flaky_translate_api, max_retries=2)

print(registry.run_tool("detect_language", text="Bonjour tout le monde"))  # -> "fr"
print(registry.run_tool("translate", text="Hello everyone"))               # -> "[ES] Hello everyone", after 1 retry
print(registry.audit_log)

In [ ]:
# A schematic LCEL-style pipe chain (prompt | model | parser) for a
# translation task. This is illustrative of the mechanism (plain operator
# overloading around function composition), not a real LangChain install
# -- a real integration would subclass a real base class, e.g. LangChain's
# `LLM` class, and implement its `_call` method.

class RunnableStep:
    def __or__(self, other):
        return PipedStep(self, other)

class PipedStep(RunnableStep):
    def __init__(self, first, second):
        self.first, self.second = first, second

    def invoke(self, x):
        return self.second.invoke(self.first.invoke(x))

class PromptStep(RunnableStep):
    def __init__(self, template):
        self.template = template

    def invoke(self, variables):
        return self.template.format(**variables)   # -> str

class LocalLLMStep(RunnableStep):
    def __init__(self, generate_fn):
        self.generate_fn = generate_fn

    def invoke(self, prompt_text):
        return self.generate_fn(prompt_text)         # -> str

class StripParserStep(RunnableStep):
    def invoke(self, raw_text):
        return raw_text.strip()                       # -> str, whitespace trimmed

translate_prompt = PromptStep("Translate to Spanish: {text}")
mock_llm = LocalLLMStep(lambda p: "  Hola a todos  ")
translate_chain = translate_prompt | mock_llm | StripParserStep()
print(translate_chain.invoke({"text": "Hello everyone"}))  # -> "Hola a todos" (whitespace stripped)

## Day 3: Local LLMs and Quantization

Cloud models trade privacy and cost-control for scale; local models trade
scale for privacy, offline use, and no per-token bill (see
`daily/Day3_...md` for the full cloud-vs-local table and the precision
bit-layout explanation). This section uses a support-ticket triage
scenario to show, in order: subprocess-isolated measurement (the
mechanism verified for real via numpy, since `transformers`/`torch`
aren't installed here), a real numpy memory-footprint measurement on an
embedding table, a CPU-only quantization fallback that actually exercises
its `except` branch in this sandbox, and a tool-call convention parsed
out of raw model text.

In [ ]:
import subprocess
import sys

def measure_alloc_in_subprocess(dtype_name, n_elements):
    """Runs one allocation in its own fresh subprocess so this dtype's
    allocator/cache state can't bleed into the next measurement -- the
    same isolation trick you'd use around
    AutoModelForCausalLM.from_pretrained(..., torch_dtype=...)."""
    script = (
        "import time, numpy as np\n"
        "t0 = time.perf_counter()\n"
        f"arr = np.zeros({n_elements}, dtype=np.{dtype_name})\n"
        "print(f'{(time.perf_counter()-t0)*1000:.3f}ms nbytes={arr.nbytes}')\n"
    )
    completed = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, timeout=30)
    return completed.stdout.strip() if completed.returncode == 0 else f"ERROR: {completed.stderr.strip()}"

for dtype in ["float32", "float16", "int8"]:
    print(dtype, "->", measure_alloc_in_subprocess(dtype, 20_000_000))
# nbytes confirms real dtype sizes: 20,000,000 elements * 4/2/1 bytes each.

# The production version of this pattern, against a real model (correct
# API shape, NOT executed in this sandbox -- transformers/torch aren't installed):
#
# def measure_model_load(model_id, dtype_name):
#     script = (
#         "import time, torch\n"
#         "from transformers import AutoModelForCausalLM\n"
#         "t0 = time.time()\n"
#         f"m = AutoModelForCausalLM.from_pretrained('{model_id}', torch_dtype=torch.{dtype_name})\n"
#         "print(f'{time.time()-t0:.2f}s')\n"
#     )
#     return subprocess.run([sys.executable, "-c", script], capture_output=True, text=True).stdout.strip()

In [ ]:
import numpy as np

# Real memory footprint of a support-ticket embedding table: a 50,000-word
# vocabulary x 768-dim embedding, measured with real numpy .nbytes.
vocab_size, embed_dim = 50_000, 768
n_params = vocab_size * embed_dim
rng = np.random.default_rng(0)

w_fp32 = rng.standard_normal((vocab_size, embed_dim)).astype(np.float32)
w_fp16 = w_fp32.astype(np.float16)                                   # real downcast, real rounding
w_int8 = np.clip(np.round(w_fp32 * 20), -127, 127).astype(np.int8)   # toy affine quant

print(f"embedding table: {vocab_size} x {embed_dim} = {n_params:,} params")
for name, arr in [("fp32", w_fp32), ("fp16", w_fp16), ("int8", w_int8)]:
    print(f"  {name}: {arr.nbytes:,} bytes ({arr.nbytes/1024**2:.2f} MiB)")
print(f"  fp32 -> int8 shrink factor: {w_fp32.nbytes / w_int8.nbytes:.2f}x")  # -> 4.00x, exactly as expected

In [ ]:
# A CPU-only fallback case study: an 8-bit quantization step that
# requires bitsandbytes + a CUDA backend, and should fail gracefully
# (not crash the whole script) when neither is available -- as they
# are not in this sandbox, so the except branch below runs for real.

def build_quantized_layer(in_features, out_features):
    try:
        import bitsandbytes as bnb
        return bnb.nn.Linear8bitLt(in_features, out_features, has_fp16_weights=False)
    except Exception as e:
        print(f"quantized layer unavailable ({type(e).__name__}: {e}); using full precision instead")
        return f"fallback_linear({in_features}x{out_features})"  # stand-in for torch.nn.Linear

layer = build_quantized_layer(512, 512)
print("layer:", layer)

In [ ]:
import re

# A tool-call convention: the model is instructed to emit lines like
#   TOOL_CALL: lookup_order_status(order_id=4471)
# which Python then extracts with a regex and dispatches. Small local
# models are far more reliable at reproducing this fixed template than
# at emitting well-formed, correctly escaped JSON.
TOOL_CALL_PATTERN = re.compile(r"TOOL_CALL:\s*(\w+)\((.*)\)")

def lookup_order_status(order_id):
    return {"order_id": order_id, "status": "shipped"}

SUPPORT_TOOLS = {"lookup_order_status": lookup_order_status}

def dispatch_from_model_text(model_text):
    match = TOOL_CALL_PATTERN.search(model_text)
    if not match:
        return None                                    # no tool call -- a valid, expected case too
    tool_name, raw_args = match.group(1), match.group(2)
    order_id = int(raw_args.split("=")[1])
    return SUPPORT_TOOLS[tool_name](order_id=order_id)

print(dispatch_from_model_text("TOOL_CALL: lookup_order_status(order_id=4471)"))  # -> {'order_id': 4471, 'status': 'shipped'}
print(dispatch_from_model_text("Let me think about this..."))                     # -> None, not an error

## Day 4: Agent Safety Mechanisms

Four guardrails -- tool allowlisting (default-deny), a step limit, a
human-approval gate, and a cost cap -- combined into one guarded runner,
checked cheapest-first (see `daily/Day4_...md` for the full decision-flow
diagram). This section uses a social-media/refund-processing agent and,
critically, runs a **correct-order** gate against a **buggy-order** gate
on the identical sequence of attempts, to show for real -- not just
assert -- that check order changes both the final budget and what gets
logged as the reason for a denial.

In [ ]:
import time

class SafeAgentGate:
    """Combines four guardrails, checked cheapest/fastest first, and --
    critically -- only COMMITS step_count/spent_usd in the final `allowed`
    branch, after every check has already passed."""

    def __init__(self, allowed_tools, max_steps, budget_usd, approve_fn):
        self.allowed_tools = set(allowed_tools)
        self.max_steps = max_steps
        self.budget_usd = budget_usd
        self.spent_usd = 0.0
        self.steps_taken = 0
        self.approve_fn = approve_fn
        self.audit_rows = []

    def attempt(self, tool_name, args, est_cost, needs_approval=False):
        row = {"timestamp": round(time.time(), 3), "tool": tool_name, "args": str(args),
               "cost": est_cost, "decision": None}
        if self.steps_taken >= self.max_steps:
            row["decision"] = "denied_step_limit"
        elif self.spent_usd + est_cost > self.budget_usd:
            row["decision"] = "denied_cost_cap"
        elif tool_name not in self.allowed_tools:
            row["decision"] = "denied_not_allowlisted"
        elif needs_approval and not self.approve_fn(tool_name, args):
            row["decision"] = "denied_no_approval"
        else:
            row["decision"] = "allowed"
            self.steps_taken += 1
            self.spent_usd += est_cost      # only committed once every check passed
        self.audit_rows.append(row)
        return row["decision"] == "allowed"


class BuggySafeAgentGate(SafeAgentGate):
    """Same four checks, WRONG order: cost is committed before the
    allowlist check runs, so a disallowed tool's estimated cost still
    gets charged to the budget even though the call never executes."""

    def attempt(self, tool_name, args, est_cost, needs_approval=False):
        row = {"timestamp": round(time.time(), 3), "tool": tool_name, "args": str(args),
               "cost": est_cost, "decision": None}
        if self.spent_usd + est_cost > self.budget_usd:
            row["decision"] = "denied_cost_cap"
            self.audit_rows.append(row)
            return False
        self.spent_usd += est_cost   # BUG: committed before the allowlist check below
        if tool_name not in self.allowed_tools:
            row["decision"] = "denied_not_allowlisted"
            self.audit_rows.append(row)
            return False
        row["decision"] = "allowed"
        self.steps_taken += 1
        self.audit_rows.append(row)
        return True


def post_to_social_media(caption):
    return f"posted: {caption}"

def refund_customer(order_id, amount):
    return f"refunded {amount} for order {order_id}"

def always_deny(tool_name, args):
    return False  # stand-in for a real human-approval prompt

In [ ]:
# A stuck agent retrying the exact same disallowed call twice, then one
# legitimate allowed call -- run through BOTH gates on identical input.

def run_scenario(gate_cls):
    gate = gate_cls(allowed_tools={"post_to_social_media"}, max_steps=10, budget_usd=1.00, approve_fn=always_deny)
    gate.attempt("refund_customer", {"order_id": 91, "amount": 0.40}, est_cost=0.40, needs_approval=True)
    gate.attempt("refund_customer", {"order_id": 91, "amount": 0.40}, est_cost=0.40, needs_approval=True)
    gate.attempt("post_to_social_media", {"caption": "New product!"}, est_cost=0.02)
    return gate

correct_gate = run_scenario(SafeAgentGate)
buggy_gate = run_scenario(BuggySafeAgentGate)

print("=== correct order ===")
for r in correct_gate.audit_rows:
    print(r)
print(f"spent_usd = {correct_gate.spent_usd:.2f}\n")

print("=== buggy order (cost committed before allowlist check) ===")
for r in buggy_gate.audit_rows:
    print(r)
print(f"spent_usd = {buggy_gate.spent_usd:.2f}")

assert round(correct_gate.spent_usd, 2) == 0.02   # only the allowed post ever spent anything
assert round(buggy_gate.spent_usd, 2) == 0.82      # two never-allowed refund attempts still "spent" 0.40 each
print("\nconfirmed: buggy order over-counts spend for calls that were never actually allowed")

In [ ]:
import pandas as pd

audit_df = pd.DataFrame(correct_gate.audit_rows)

# Flag (tool, args) combinations attempted an unusually high number of
# times -- a simple heuristic for a stuck or misbehaving agent, no ML
# model required. Here the repeated, identical, denied refund_customer
# attempt is exactly the kind of thing this should catch.
repeat_counts = (
    audit_df.groupby(["tool", "args"])
    .size()
    .reset_index(name="attempts")
)
flagged = repeat_counts[repeat_counts["attempts"] >= 2]
print(audit_df)
print("\nflagged:\n", flagged)

# Model-routing pattern (qualitative, no invented figures): route routine
# steps to a cheap local model and escalate to a larger cloud model only
# for one best-effort attempt once the local agent is stuck or out of
# steps -- most requests stay on the cheaper path, and hard cases still
# get a shot at a more capable model. The escalation call still passes
# through the same guardrails above -- being the "expensive fallback" is
# not a reason to skip the cost cap.
def route_request(local_agent_run, cloud_agent_run, question):
    local_result = local_agent_run(question)
    if local_result is None:
        return cloud_agent_run(question)  # escalate only when stuck
    return local_result